In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# PeptiScout AI Final Notebook

**Course:** CPS 5801 Advanced AI Final Project  
**Project:** LLM/VLM-Driven AI Agent for Peptide Research QA, Reconstitution, Safety, and Citation Auditing

This notebook is the packaged final notebook artifact for the CPS 5801 technical requirements. It excludes the IEEE paper, presentation slides/video, and teammate report, but covers Tasks 1-4: problem definition, system architecture, baseline LLM/VLM, lightweight fine-tuning, tool-based agent design, quantitative evaluation, and ablations.

The final experiment is designed as a same-model comparison using `meta-llama/Llama-3.2-11B-Vision-Instruct` loaded in 4-bit quantized form.

| Approach | Model setup | Tools |
| --- | --- | --- |
| Baseline LLM/VLM | Prompt-only Llama, no adapter | No tools |
| Fine-tuned LLM/VLM | Same Llama base + 4-bit QLoRA adapter | No tools |
| Tool-based agent | Same Llama base with tool context | Calculator, RAG/web/VLM context where available |

Colab execution does **not** require FastAPI. This notebook loads JSON files, runs inference directly, and scores predictions directly.

## Task 1: Problem Definition, Architecture, and Evaluation Plan

**Application scenario:** PeptiScout AI assists with research-context peptide questions, including reconstitution math, mechanism-of-action summaries, safety cautions, citation auditing, and optional bloodwork image context.

**Target users:** students, peptide researchers, and research reviewers. The system is not intended to provide medical advice.

**Inputs:** natural-language peptide questions, optional vial/dose/syringe details, optional vendor/source questions, and optional bloodwork image inputs.

**Outputs:** a structured four-part response: `protocol`, `moa`, `good_bad`, and `audit_trail`, with optional `react_trace` for agent runs.

**Task type:** QA/reasoning with deterministic dosage calculation, source/citation validation, retrieval-augmented synthesis, and VLM-assisted image interpretation.

**Evaluation benchmark:** `benchmark_100.json`, containing dosage, MOA, safety, and vendor/source queries.

**Metrics:** DS strict/loose for dosage, PC for pathway/cofactor completeness, TSR for treatment safety/risk guidance, and CA for citation accuracy through PMID validation.

## System Architecture

Full agent flow:

```text
router -> proactive_check -> dose_rag -> dose_tavily -> dose_extractor -> calculator -> rag_retriever -> vlm_analyzer -> source_vetter -> synthesizer
```

**Tool A:** deterministic calculator / optimal reconstitution.  
**Tool B:** Pinecone RAG over PubMed-style chunks.  
**Tool C:** VLM bloodwork analyzer.  
**Tool D:** Tavily source/web vetting and dosing research.

For the Colab no-FastAPI evaluation, the same comparison idea is preserved by using direct notebook functions: prompt-only Llama, fine-tuned Llama, and a tool-context Llama prompt that injects deterministic calculator outputs and available benchmark/tool context.

## Colab Runtime Setup

Run this notebook in Google Colab with an A100 GPU for the 4-bit QLoRA section:

1. Runtime > Change runtime type > A100 GPU.
2. Put the project files in Google Drive under `/content/drive/MyDrive/Pepti_scout/`:
   - `peptide_dataset.json`
   - `benchmark_100.json`
   - optional `results.json`
   - optional `run_evaluation.py`
3. Make sure your Hugging Face token has access to `meta-llama/Llama-3.2-11B-Vision-Instruct`.

The notebook uses network access for Hugging Face downloads and optional NCBI PMID validation. Before final submission, run the final inference/export cells and save the notebook with outputs visible.

In [ ]:
# Colab-only dependency setup. Run this cell in Colab, not necessarily on your local machine.
import sys, subprocess, os

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "transformers>=4.45.0", "accelerate", "peft", "bitsandbytes", "trl", "datasets", "pandas", "requests"
    ])
    print("Colab dependencies installed.")
else:
    print("Not running in Colab; skipping package installation.")

Colab dependencies installed.


In [ ]:
from __future__ import annotations

import json
import math
import os
import re
import statistics
import sys
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd
import requests

MODEL_ID = "meta-llama/Llama-3.2-11B-Vision-Instruct"
QUANTIZATION = "4bit-nf4"
IN_COLAB = "google.colab" in sys.modules
COLAB_PROJECT_DIR = Path("/content/drive/MyDrive/Pepti_scout")
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    COLAB_PROJECT_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_OUT = COLAB_PROJECT_DIR / "results_llama_11b.json" if COLAB_PROJECT_DIR.exists() else Path("results_llama_11b.json")
PREDICTIONS_OUT = COLAB_PROJECT_DIR / "predictions_llama_11b.json" if COLAB_PROJECT_DIR.exists() else Path("predictions_llama_11b.json")

def find_data_file(name: str) -> Path:
    candidates = [
        Path(name),
        COLAB_PROJECT_DIR / name,
        Path("/content") / name,
        Path("../data") / name,
        Path("../../backend/data") / name,
        Path("backend/data") / name,
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(f"Could not find {name}. Upload it to Colab or run from repo root.")

benchmark_path = find_data_file("benchmark_100.json")
dataset_path = find_data_file("peptide_dataset.json")
benchmark = json.loads(benchmark_path.read_text())
instruction_data = json.loads(dataset_path.read_text())

print(f"Loaded {len(benchmark)} benchmark rows from {benchmark_path}")
print(f"Loaded {len(instruction_data)} instruction examples from {dataset_path}")
pd.Series([row.get("category") for row in benchmark]).value_counts().rename("count").to_frame()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded 100 benchmark rows from /content/drive/MyDrive/Pepti_scout/benchmark_100.json
Loaded 1071 instruction examples from /content/drive/MyDrive/Pepti_scout/peptide_dataset.json


,count
dosage,40
moa,30
safety,20
vendor,10


## Scoring Helpers

These helpers mirror the evaluation script while adding robustness for structured numeric fields such as `recommended_dose_mcg`, `bac_water_mL`, and `syringe_units`. Structured fields are preferred before regex extraction, which reduces scoring errors when a model mentions both vial mass and dose.

In [ ]:
PMID_RE = re.compile(r"\b(\d{7,8})\b")
NCBI_ESUMMARY = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi"
NCBI_SLEEP_S = 0.34

TSR_GROUP_SAFETY = (
    "contraindication", "contraindicated", "avoid", "caution", "warning", "adverse",
    "risk", "side effect", "monitor", "do not use", "precaution",
)
TSR_GROUP_GUIDANCE = (
    "recommend", "advised", "suggested", "protocol", "administer",
    "guideline", "use", "dosing", "schedule",
)

def safe_float(value):
    if value is None:
        return None
    try:
        return float(value)
    except (TypeError, ValueError):
        return None

def normalize_response(raw: Any) -> dict[str, Any]:
    if isinstance(raw, dict):
        return {
            "protocol": str(raw.get("protocol", "")),
            "moa": str(raw.get("moa", "")),
            "good_bad": str(raw.get("good_bad", "")),
            "audit_trail": str(raw.get("audit_trail", "")),
            "react_trace": raw.get("react_trace"),
            **{k: v for k, v in raw.items() if k not in {"protocol", "moa", "good_bad", "audit_trail", "react_trace"}},
        }
    text = str(raw or "").strip()
    try:
        start, end = text.find("{"), text.rfind("}")
        if start != -1 and end > start:
            return normalize_response(json.loads(text[start:end + 1]))
    except Exception:
        pass
    return {"protocol": text, "moa": "", "good_bad": "", "audit_trail": "", "react_trace": None}

def structured_number(raw: dict[str, Any], names: tuple[str, ...]) -> float | None:
    for name in names:
        if name in raw:
            val = safe_float(raw.get(name))
            if val is not None:
                return val
    protocol = raw.get("protocol")
    if isinstance(protocol, dict):
        for name in names:
            val = safe_float(protocol.get(name))
            if val is not None:
                return val
    return None

def extract_predicted_dose_mcg(raw: dict[str, Any]) -> tuple[float | None, str | None]:
    direct = structured_number(raw, ("recommended_dose_mcg", "dose_mcg", "dose", "recommendedDoseMcg"))
    if direct is not None:
        return direct, None
    text = str(raw.get("protocol", ""))
    patterns = [
        r"(?:recommended\s+)?dose(?:\s+is|\s+of|:)?\s*(\d+\.?\d*)\s*(?:mcg|ug|µg)\b(?!\s*/\s*mL)",
        r"(?:administer|inject|take|use)\s*(\d+\.?\d*)\s*(?:mcg|ug|µg)\b(?!\s*/\s*mL)",
    ]
    for pattern in patterns:
        match = re.search(pattern, text, flags=re.I)
        if match:
            return safe_float(match.group(1)), None
    return None, "no_dose_pattern"

def extract_predicted_water_ml(raw: dict[str, Any]) -> tuple[float | None, str | None]:
    direct = structured_number(raw, (
        "bac_water_mL", "bac_water_ml", "BAC_water_amount", "water_mL", "water_ml",
        "reconstitution_water_mL", "reconstitution_water_ml", "water_amount_ml",
    ))
    if direct is not None:
        return direct, None
    text = str(raw.get("protocol", ""))
    patterns = [
        r"(?:BAC_water_amount|bac_water_mL|bac_water_ml|water_mL|water_ml|reconstitution_water_mL|water_amount_ml)['\"]?\s*:\s*['\"]?(\d+\.?\d*)",
        r"(?:reconstitute|reconstitution|add|mix|dilute|use)\D{0,40}(\d+\.?\d*)\s*mL\D{0,40}(?:BAC|bacteriostatic|water)",
        r"(\d+\.?\d*)\s*mL\D{0,40}(?:BAC|bacteriostatic|water)",
        r"(?:BAC|bacteriostatic|water)\D{0,40}(\d+\.?\d*)\s*mL",
    ]
    candidates = []
    for pattern in patterns:
        for match in re.finditer(pattern, text, flags=re.I):
            val = safe_float(match.group(1))
            if val is not None and 0.25 <= val <= 5.0:
                candidates.append(val)
    if candidates:
        return candidates[0], None
    return None, "no_water_pattern"

def score_ds(raw: dict[str, Any], entry: dict[str, Any]) -> tuple[int | None, int | None, str | None]:
    gold_dose_range = entry.get("gold_dose_range")
    gold_water_range = entry.get("gold_water_range")
    if isinstance(gold_dose_range, list) and len(gold_dose_range) == 2 and isinstance(gold_water_range, list) and len(gold_water_range) == 2:
        pred_dose, dose_note = extract_predicted_dose_mcg(raw)
        pred_water, water_note = extract_predicted_water_ml(raw)
        if pred_dose is None or pred_water is None:
            return None, None, ";".join(n for n in (dose_note, water_note) if n) or "range_prediction_missing"
        ds_dose = int(float(gold_dose_range[0]) <= pred_dose <= float(gold_dose_range[1]))
        ds_water = int(float(gold_water_range[0]) <= pred_water <= float(gold_water_range[1]))
        return int(ds_dose and ds_water), ds_water, None
    return None, None, "no_gold_range"

def score_pc(full_response: str, gold_cofactors: list[str]) -> float | None:
    if not gold_cofactors:
        return None
    low = full_response.lower()
    return sum(1 for c in gold_cofactors if str(c).lower() in low) / len(gold_cofactors)

def score_tsr(full_response: str) -> int:
    low = full_response.lower()
    return int(any(w in low for w in TSR_GROUP_SAFETY) and any(w in low for w in TSR_GROUP_GUIDANCE))

def extract_pmids(audit_trail: str) -> list[str]:
    return list(dict.fromkeys(PMID_RE.findall(audit_trail or "")))

def validate_pmid(pmid: str) -> bool:
    try:
        response = requests.get(NCBI_ESUMMARY, params={"db": "pubmed", "id": pmid, "retmode": "json"}, timeout=45)
        response.raise_for_status()
        result = response.json().get("result", {})
        return pmid in result and not result.get(pmid, {}).get("error")
    except Exception:
        return False

def score_ca(audit_trail: str, skip_ca: bool = False) -> tuple[float | None, str | None]:
    if skip_ca:
        return None, "skipped"
    pmids = extract_pmids(audit_trail)
    if not pmids:
        return None, "no_pmids"
    values = []
    for i, pmid in enumerate(pmids):
        if i:
            time.sleep(NCBI_SLEEP_S)
        values.append(1.0 if validate_pmid(pmid) else 0.0)
    return sum(values) / len(values), None

def score_predictions(predictions: list[dict[str, Any]], benchmark_rows: list[dict[str, Any]], skip_ca: bool = False) -> list[dict[str, Any]]:
    bench_by_id = {row.get("id"): row for row in benchmark_rows}
    scored = []
    for pred in predictions:
        entry = bench_by_id.get(pred.get("id"))
        if not entry:
            continue
        raw = normalize_response(pred.get("response", pred))
        full_response = raw["protocol"] + raw["moa"] + raw["good_bad"] + raw["audit_trail"]
        ds_strict, ds_loose, ds_note = score_ds(raw, entry)
        ca, ca_note = score_ca(raw["audit_trail"], skip_ca=skip_ca)
        row = {
            "id": entry.get("id"),
            "category": entry.get("category"),
            "mode": pred.get("mode"),
            "scores": {
                "ds_strict": ds_strict,
                "ds_loose": ds_loose,
                "pc": score_pc(full_response, entry.get("gold_cofactors") or []),
                "tsr": score_tsr(full_response),
                "ca": ca,
            },
            "score_notes": "; ".join(n for n in (f"ds:{ds_note}" if ds_note else None, f"ca:{ca_note}" if ca_note else None) if n),
        }
        scored.append(row)
    return scored

def mean_skip_null(values):
    vals = [float(v) for v in values if v is not None]
    return float(statistics.mean(vals)) if vals else None

def build_comparison_table(per_query: list[dict[str, Any]]) -> dict[str, dict[str, Any]]:
    modes = list(dict.fromkeys(row["mode"] for row in per_query))
    out = {}
    for mode in modes:
        rows = [row for row in per_query if row["mode"] == mode]
        out[mode] = {
            "mean_ds": mean_skip_null([row["scores"].get("ds_loose") for row in rows]),
            "mean_ds_strict": mean_skip_null([row["scores"].get("ds_strict") for row in rows]),
            "mean_pc": mean_skip_null([row["scores"].get("pc") for row in rows]),
            "mean_tsr": mean_skip_null([row["scores"].get("tsr") for row in rows]),
            "mean_ca": mean_skip_null([row["scores"].get("ca") for row in rows]),
            "n": len(rows),
        }
    return out

## Task 2: Prompt-Only Llama Baselines

The baseline system uses `meta-llama/Llama-3.2-11B-Vision-Instruct` without LoRA adapters and without tools. The three prompt settings correspond to zero-shot, few-shot, and chain-of-thought-style planning prompts while still requiring a structured JSON answer.

In [ ]:
BASE_SYSTEM = """You are PeptiScout, an expert AI assistant specializing in research peptides.
When a user asks about a peptide, respond as JSON with these keys:
- protocol: dosing/reconstitution guidance when relevant, including numeric recommended_dose_mcg, bac_water_mL, and syringe_units when possible
- moa: mechanism of action and pathway-level explanation
- good_bad: benefits plus contraindications, cautions, warnings, adverse effects, and research-use limitations
- audit_trail: cite PubMed studies by PMID when known; do not fabricate PMIDs

Return JSON only."""

FEW_SHOT = """Example user: What is the reconstitution dose for 2mg Semax with 1mL BAC water?
Example assistant: {"protocol":"2mg in 1mL = 2000mcg/mL. A 300mcg dose = 0.15mL = 15 units U100.","recommended_dose_mcg":300,"bac_water_mL":1.0,"syringe_units":15,"moa":"Semax is an ACTH analog associated with BDNF/NGF signaling.","good_bad":"Potential cognitive/neuroprotective research interest; caution with MAOIs and limited human evidence.","audit_trail":"PMID 19230835; PMID 22750014"}

Example user: What co-factors does GHK-Cu require?
Example assistant: {"protocol":"Research protocols vary; no reconstitution math requested.","moa":"GHK-Cu is associated with collagen remodeling and inflammatory signaling modulation.","good_bad":"Potential wound-healing interest; caution with copper sensitivity and unregulated products.","audit_trail":"PMID 25170290; PMID 28759605"}"""

COT_HINT = """Before answering, internally identify the peptide, whether dosage math is needed, relevant pathway/cofactors, contraindications, and citation support. Do not reveal hidden reasoning; return JSON only."""

def build_messages(question: str, mode: str):
    if mode == "llama-baseline-few-shot":
        return [{"role": "system", "content": BASE_SYSTEM}, {"role": "user", "content": FEW_SHOT + "\n\nUser: " + question}]
    if mode == "llama-baseline-cot":
        return [{"role": "system", "content": BASE_SYSTEM + "\n\n" + COT_HINT}, {"role": "user", "content": question}]
    return [{"role": "system", "content": BASE_SYSTEM}, {"role": "user", "content": question}]

In [ ]:
from google.colab import userdata
import os

os.environ["HF_TOKEN"] = userdata.get("llamatoken")

In [ ]:
# Load the 4-bit quantized Llama Vision-Instruct model.
# This can take several minutes and requires Hugging Face access to the model.
import torch
from transformers import AutoProcessor, BitsAndBytesConfig, MllamaForConditionalGeneration

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

processor = AutoProcessor.from_pretrained(MODEL_ID)
base_model = MllamaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
base_model.eval()
print("Loaded", MODEL_ID, "with", QUANTIZATION)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/906 [00:00<?, ?it/s]

Loaded meta-llama/Llama-3.2-11B-Vision-Instruct with 4bit-nf4


In [ ]:
def generate_text(model, messages, max_new_tokens=768, temperature=0.2):
    prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=temperature > 0,
            temperature=temperature,
            top_p=0.9,
            pad_token_id=processor.tokenizer.eos_token_id,
        )
    new_tokens = output[0, inputs["input_ids"].shape[-1]:]
    return processor.decode(new_tokens, skip_special_tokens=True)

def run_prompt_mode(model, rows, mode: str, limit: int | None = None):
    predictions = []
    selected = rows[:limit] if limit else rows
    for row in selected:
        raw = generate_text(model, build_messages(row["query"], mode))
        predictions.append({"id": row["id"], "mode": mode, "response": normalize_response(raw), "raw_text": raw})
    return predictions

# Start with a small smoke test, then set limit=None for the final run.
# baseline_predictions = []
# for mode in ["llama-baseline-zero-shot", "llama-baseline-few-shot", "llama-baseline-cot"]:
#     baseline_predictions.extend(run_prompt_mode(base_model, benchmark, mode, limit=3))

## Task 3 Part 1: 4-bit QLoRA Fine-Tuning

This section fine-tunes LoRA adapters on top of the 4-bit quantized Llama base model. The base model remains quantized and frozen; only small adapter weights are trained. This satisfies the lightweight model adaptation requirement while fitting in Colab A100 memory.

In [ ]:
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer

def to_training_text(example: dict[str, Any]) -> dict[str, str]:
    user = example.get("instruction", "")
    if example.get("input"):
        user += "\n" + str(example["input"])
    assistant_text = example.get("output", "")
    messages = [
        {"role": "system", "content": BASE_SYSTEM},
        {"role": "user", "content": user},
        {"role": "assistant", "content": assistant_text},
    ]
    return {"text": processor.apply_chat_template(messages, tokenize=False)}

train_dataset = Dataset.from_list([to_training_text(x) for x in instruction_data])
train_dataset = train_dataset.shuffle(seed=42)
print(train_dataset[0]["text"][:1500])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 30 Apr 2026

You are PeptiScout, an expert AI assistant specializing in research peptides.
When a user asks about a peptide, respond as JSON with these keys:
- protocol: dosing/reconstitution guidance when relevant, including numeric recommended_dose_mcg, bac_water_mL, and syringe_units when possible
- moa: mechanism of action and pathway-level explanation
- good_bad: benefits plus contraindications, cautions, warnings, adverse effects, and research-use limitations
- audit_trail: cite PubMed studies by PMID when known; do not fabricate PMIDs

Return JSON only.<|eot_id|><|start_header_id|>user<|end_header_id|>

What are the mechanisms of action, co-factors, benefits, and risks associated with the peptide PT-141?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

[MOA]: PT-141 acts as a melanocortin receptor agonist, specifically targeting the MC1, MC3, and MC4 receptors, w

In [ ]:
qlora_model = prepare_model_for_kbit_training(base_model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

qlora_model = get_peft_model(qlora_model, lora_config)
qlora_model.print_trainable_parameters()

sft_config = SFTConfig(
    output_dir="llama_3_2_11b_peptiscout_qlora",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=10,
    save_steps=100,
    max_length=2048,
    bf16=True,
    report_to="none",
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=qlora_model,
    train_dataset=train_dataset,
    args=sft_config,
    processing_class=processor.tokenizer,
)

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


trainable params: 58,982,400 || all params: 10,729,203,235 || trainable%: 0.5497


Adding EOS to train dataset:   0%|          | 0/1071 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1071 [00:00<?, ? examples/s]

In [ ]:
trainer.train()
trainer.save_model("llama_3_2_11b_peptiscout_qlora/final_adapter")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'bos_token_id': 128000, 'pad_token_id': 128004}.


Step,Training Loss
10,1.624793
20,0.555171
30,0.537031
40,0.490017
50,0.454117
60,0.433844
70,0.434763
80,0.460054
90,0.434414
100,0.430111


## Task 3 Part 2: Tool-Based Agent Variant Without FastAPI

The production app uses LangGraph and FastAPI. In Colab, the evaluation avoids FastAPI and injects tool outputs directly into the Llama prompt. This keeps the system comparison reproducible while preserving the tool-based idea: deterministic reconstitution math and structured tool context are supplied to the same base model.

In [ ]:
def recommend_reconstitution(vial_mg: float, dose_mcg: float, syringe_type: str = "U100") -> dict[str, Any]:
    units_per_ml = 100 if syringe_type == "U100" else 40
    target_units = 10
    target_inject_ml = target_units / units_per_ml
    ideal_water_ml = target_inject_ml * vial_mg * 1000 / dose_mcg
    water_ml = min(3.0, max(1.0, round(ideal_water_ml * 2) / 2))
    concentration = vial_mg * 1000 / water_ml
    inject_ml = dose_mcg / concentration
    syringe_units = inject_ml * units_per_ml
    return {
        "vial_mg": vial_mg,
        "recommended_dose_mcg": dose_mcg,
        "bac_water_mL": water_ml,
        "concentration_mcg_per_mL": concentration,
        "injection_mL": inject_ml,
        "syringe_units": syringe_units,
        "syringe_type": syringe_type,
    }

STANDARD_DOSE_MCG = {
    "bpc-157": 250, "tb-500": 1000, "thymosin beta-4": 1000,
    "semax": 500, "selank": 300, "ghk-cu": 1500,
    "ipamorelin": 200, "cjc-1295": 200, "epithalon": 5000,
    "pt-141": 1500, "dsip": 200, "hexarelin": 150,
    "ghrp-6": 200, "melanotan": 300, "tesamorelin": 1000,
    "sermorelin": 300, "aod-9604": 300, "igf-1 lr3": 100,
    "mgf": 150, "kisspeptin": 250,
}

def researched_standard_dose(entry: dict[str, Any]) -> float | None:
    # No benchmark gold labels are used here. This fixed lookup represents the researched
    # dosing prior that production PeptiScout derives dynamically from RAG + Tavily.
    peptide = str(entry.get("peptide") or "").lower().strip()
    return STANDARD_DOSE_MCG.get(peptide)

def build_tool_agent_messages(entry: dict[str, Any]):
    tool_context = {}
    if entry.get("category") == "dosage" and entry.get("vial_mg"):
        dose = researched_standard_dose(entry)
        if dose:
            tool_context["calculator"] = recommend_reconstitution(float(entry["vial_mg"]), dose)
    system = BASE_SYSTEM + "\nUse the supplied tool_context when present. Preserve numeric fields exactly."
    user = json.dumps({"question": entry["query"], "tool_context": tool_context}, ensure_ascii=False)
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]

def run_tool_agent_mode(model, rows, mode="llama-full-agent", limit: int | None = None):
    predictions = []
    selected = rows[:limit] if limit else rows
    for row in selected:
        raw = generate_text(model, build_tool_agent_messages(row))
        response = normalize_response(raw)
        predictions.append({"id": row["id"], "mode": mode, "response": response, "raw_text": raw})
    return predictions

## Task 4: Evaluation and Ablation Study

Run the following cells after baseline inference, QLoRA training/inference, and tool-agent inference. The final table should contain same-Llama rows for prompt-only baseline, fine-tuned Llama, and Llama tool-agent.

In [ ]:
# Final inference control.
# Keep RUN_FINAL_INFERENCE=False while testing setup. For final submission, set it to True,
# set FINAL_RUN_LIMIT=None, run this cell, and save the notebook with outputs visible.
RUN_FINAL_INFERENCE = False
FINAL_RUN_LIMIT = 3  # use 3 for smoke testing; use None for the full benchmark
RUN_CA_VALIDATION = False  # set True for final CA validation through NCBI network calls

if RUN_FINAL_INFERENCE:
    all_predictions = []
    for mode in ["llama-baseline-zero-shot", "llama-baseline-few-shot", "llama-baseline-cot"]:
        all_predictions.extend(run_prompt_mode(base_model, benchmark, mode, limit=FINAL_RUN_LIMIT))
    all_predictions.extend(run_prompt_mode(qlora_model, benchmark, "llama-fine-tuned", limit=FINAL_RUN_LIMIT))
    all_predictions.extend(run_tool_agent_mode(base_model, benchmark, "llama-full-agent", limit=FINAL_RUN_LIMIT))
    PREDICTIONS_OUT.write_text(json.dumps(all_predictions, indent=2, ensure_ascii=False))
    print(f"Wrote {len(all_predictions)} predictions to {PREDICTIONS_OUT}")
elif PREDICTIONS_OUT.exists():
    all_predictions = json.loads(PREDICTIONS_OUT.read_text())
    print(f"Loaded existing predictions from {PREDICTIONS_OUT}: {len(all_predictions)} rows")
else:
    print("RUN_FINAL_INFERENCE is False and no predictions file exists; using scoring smoke test only.")
    all_predictions = [
        {
            "id": benchmark[0]["id"],
            "mode": "scoring-smoke-test",
            "response": {
                "protocol": "Recommended dose is 250 mcg. Reconstitute with 2.0 mL BAC water. Draw 10 units on U100.",
                "recommended_dose_mcg": 250,
                "bac_water_mL": 2.0,
                "syringe_units": 10,
                "moa": "Research context.",
                "good_bad": "Use caution; monitor adverse effects. Suggested research protocol only.",
                "audit_trail": "",
            },
        }
    ]

pd.DataFrame(all_predictions)[["id", "mode"]].head()

RUN_FINAL_INFERENCE is False and no predictions file exists; using scoring smoke test only.


,id,mode
0,1,scoring-smoke-test


In [ ]:
# Score active predictions and export a results JSON. For final submission, run this after
# RUN_FINAL_INFERENCE=True has generated full predictions.
per_query = score_predictions(all_predictions, benchmark, skip_ca=not RUN_CA_VALIDATION)
comparison_table = build_comparison_table(per_query)
results_doc = {
    "meta": {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "base_model": MODEL_ID,
        "quantization": QUANTIZATION,
        "adapter_path": "llama_3_2_11b_peptiscout_qlora/final_adapter",
        "model_family": "llama",
        "fastapi_used": False,
        "ca_validation_enabled": RUN_CA_VALIDATION,
        "benchmark_path": str(benchmark_path),
        "predictions_path": str(PREDICTIONS_OUT),
    },
    "comparison_table": comparison_table,
    "per_query": per_query,
}
RESULTS_OUT.write_text(json.dumps(results_doc, indent=2, ensure_ascii=False))
print(f"Wrote scored results to {RESULTS_OUT}")
pd.DataFrame(comparison_table).T

Wrote scored results to /content/drive/MyDrive/Pepti_scout/results_llama_11b.json


,mean_ds,mean_ds_strict,mean_pc,mean_tsr,mean_ca,n
scoring-smoke-test,1.0,1.0,NaN,1.0,NaN,1.0


## Ablation Interpretation

After the final same-Llama run, the next cell computes ablation deltas from `per_query` and prints interpretation text. If you only ran the smoke test or did not generate a specific ablation mode, that comparison will show as unavailable.

Target comparisons:

- **A: Calculator contribution:** `llama-full-agent` vs `llama-full-agent-no-calculator` on dosage DS loose.
- **B: Reasoning/tool context contribution:** `llama-full-agent` vs prompt-only Llama on MOA PC.
- **C: Fine-tuning vs retrieval/tool context:** `llama-fine-tuned` vs `llama-rag-only-no-finetune`, if available.

Discussion should address limitations: small benchmark size, safety constraints, possible PMID hallucination, extraction sensitivity, and the fact that peptide dosing outputs are research-only and not medical advice.

In [ ]:
def mean_metric(mode: str, metric: str, category: str | None = None):
    values = []
    for row in per_query:
        if row.get("mode") != mode:
            continue
        if category and row.get("category") != category:
            continue
        value = row.get("scores", {}).get(metric)
        if value is not None:
            values.append(float(value))
    return statistics.mean(values) if values else None

def fmt_pct(value):
    return "not available" if value is None else f"{value * 100:.1f}%"

def fmt_delta(value):
    return "not available" if value is None else f"{value * 100:+.1f} percentage points"

full_ds = mean_metric("llama-full-agent", "ds_loose", "dosage")
no_calc_ds = mean_metric("llama-full-agent-no-calculator", "ds_loose", "dosage")
delta_a = full_ds - no_calc_ds if full_ds is not None and no_calc_ds is not None else None

full_pc = mean_metric("llama-full-agent", "pc", "moa")
baseline_pc_candidates = {
    mode: mean_metric(mode, "pc", "moa")
    for mode in ["llama-baseline-zero-shot", "llama-baseline-few-shot", "llama-baseline-cot"]
}
baseline_pc_values = [v for v in baseline_pc_candidates.values() if v is not None]
best_baseline_pc = max(baseline_pc_values) if baseline_pc_values else None
delta_b = full_pc - best_baseline_pc if full_pc is not None and best_baseline_pc is not None else None

ft_pc = mean_metric("llama-fine-tuned", "pc")
rag_pc = mean_metric("llama-rag-only-no-finetune", "pc")
delta_c_pc = ft_pc - rag_pc if ft_pc is not None and rag_pc is not None else None
ft_ca = mean_metric("llama-fine-tuned", "ca")
rag_ca = mean_metric("llama-rag-only-no-finetune", "ca")
delta_c_ca = ft_ca - rag_ca if ft_ca is not None and rag_ca is not None else None

ablation_summary = pd.DataFrame([
    {
        "ablation": "A: calculator contribution",
        "before": fmt_pct(no_calc_ds),
        "after": fmt_pct(full_ds),
        "delta": fmt_delta(delta_a),
        "interpretation": "Positive delta means deterministic calculator context improved dosage water-volume correctness.",
    },
    {
        "ablation": "B: full-agent vs best prompt baseline",
        "before": fmt_pct(best_baseline_pc),
        "after": fmt_pct(full_pc),
        "delta": fmt_delta(delta_b),
        "interpretation": "Positive delta means tool/reasoning context improved MOA cofactor/pathway completeness.",
    },
    {
        "ablation": "C: fine-tuned vs RAG-only no-finetune",
        "before": f"PC {fmt_pct(rag_pc)} / CA {fmt_pct(rag_ca)}",
        "after": f"PC {fmt_pct(ft_pc)} / CA {fmt_pct(ft_ca)}",
        "delta": f"PC {fmt_delta(delta_c_pc)} / CA {fmt_delta(delta_c_ca)}",
        "interpretation": "Unavailable values mean this optional variant was not run in the final Colab evaluation.",
    },
])

ablation_summary

,ablation,before,after,delta,interpretation
0,A: calculator contribution,not available,not available,not available,Positive delta means deterministic calculator ...
1,B: full-agent vs best prompt baseline,not available,not available,not available,Positive delta means tool/reasoning context im...
2,C: fine-tuned vs RAG-only no-finetune,PC not available / CA not available,PC not available / CA not available,PC not available / CA not available,Unavailable values mean this optional variant ...


#load Adapter

In [ ]:
from peft import PeftModel

adapter_path = "/content/llama_3_2_11b_peptiscout_qlora/final_adapter"
qlora_model = PeftModel.from_pretrained(base_model, adapter_path)
qlora_model.eval()

print("Loaded trained adapter:", adapter_path)

Loaded trained adapter: /content/llama_3_2_11b_peptiscout_qlora/final_adapter


In [ ]:
import shutil
from pathlib import Path

src = Path("/content/llama_3_2_11b_peptiscout_qlora/final_adapter")
dst = Path("/content/drive/MyDrive/Pepti_scout/llama_3_2_11b_peptiscout_qlora/final_adapter")

dst.parent.mkdir(parents=True, exist_ok=True)

if dst.exists():
    shutil.rmtree(dst)

shutil.copytree(src, dst)

print("Copied adapter to Drive:", dst)

Copied adapter to Drive: /content/drive/MyDrive/Pepti_scout/llama_3_2_11b_peptiscout_qlora/final_adapter


In [ ]:
adapter_path = "/content/drive/MyDrive/Pepti_scout/llama_3_2_11b_peptiscout_qlora/final_adapter"